In [1]:
import json 

In [2]:
with open('../municipis.geojson') as f:
    data = json.load(f)

In [3]:
m1 = data["features"][0]

In [4]:
from unidecode import unidecode

In [5]:
municipis_interes = [
    'Badalona', 
    'Castelldefels', 
    'Cornella de Llobregat', 
    'Gava', 
    'Montcada i reixac', 
    'Sant Boi de Llobregat',
    'Sant Cugat del Valles'
]
municipis_interes = [''.join(m.split(" ")).lower() for m in municipis_interes]
municipis_interes

['badalona',
 'castelldefels',
 'cornelladellobregat',
 'gava',
 'montcadaireixac',
 'santboidellobregat',
 'santcugatdelvalles']

In [6]:
munIds = {m : i+11 for (i,m) in enumerate(municipis_interes)}
munIds["montcadairexac"] = 15

In [7]:
features_interes = []
for feat in data["features"]:
    nom_muni = feat["properties"]["nom_muni"]
    nom_muni_norm = unidecode("".join(nom_muni.split(" ")).lower())
    if nom_muni_norm in municipis_interes:
        features_interes.append(feat)

In [8]:
features_interes[0].keys()

dict_keys(['type', 'properties', 'geometry'])

In [9]:
rows = []
with open("../municipis_demo_info.txt", 'r') as file:
    text = file.read()
    lines = text.split('\n')
    for line in lines:
        rows.append(line.replace(',','').split(';'))
fields = rows[0]
rows = rows[1:]

In [10]:
muni_info_all = {}
for row in rows:
    nom_muni = row[0]
    nom_muni_norm = unidecode("".join(nom_muni.split(" ")).lower()) 
    if nom_muni_norm == 'barcelona': 
        continue
    muni_id = munIds[nom_muni_norm]
    year = row[1]
    muni_year_info = {}
    for (k,v) in zip(fields,row): 
        muni_year_info[k] = v
    if muni_id not in muni_info_all:
        muni_info_all[muni_id] = {}
    muni_info_all[muni_id][year] = muni_year_info


In [11]:
import copy
full_geojson = copy.deepcopy(data)


In [ ]:
full_features_interes = []
for feat in features_interes: 
    nom_muni = feat["properties"]["nom_muni"]
    nom_muni_norm = unidecode("".join(nom_muni.split(" ")).lower())
    muni_id = munIds[nom_muni_norm]
    muni_info = muni_info_all[muni_id]
    feat["properties"]["MUNI_ID"] = muni_id
    feat["properties"]["DEMO_INFO"] = muni_info
    feat["properties"]["SCONJ_DESC"] = "Municipi"
    full_features_interes.append(feat)



In [ ]:
full_geojson["features"] = full_features_interes

In [ ]:
with open('../MUNI_INFO.json', 'w') as fp:
	json.dump(full_geojson, fp)